# 01 · Phân loại văn bản

**Các bước:**
1. Đọc dữ liệu, chia train / valid
2. Baseline nhanh: TF-IDF + Logistic Regression
3. Fine-tune Transformer
4. Kết hợp 2 model → nộp bài

Lý thuyết: xem `knowledge/01_phan_loai_van_ban.md`

In [ ]:
# !pip install -U transformers datasets accelerate scikit-learn pyvi

## 0. Import

In [ ]:
import re
import unicodedata

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 1. Cấu hình

In [ ]:
TRAIN_FILE = None     # vd "train.csv"   (None = dùng dữ liệu mẫu)
TEST_FILE = None      # vd "test.csv"
TEXT_COL = "text"
LABEL_COL = "label"

MODEL_NAME = "xlm-roberta-base"   # tiếng Việt có thể thử "vinai/phobert-base-v2" (nhớ bật USE_WORD_SEG)
USE_WORD_SEG = False              # True nếu dùng PhoBERT
MAX_LEN = 256

## 2. Đọc dữ liệu

In [ ]:
def load_demo_data():
    from datasets import load_dataset
    data = load_dataset("dair-ai/emotion")
    train = data["train"].to_pandas()
    test = data["test"].to_pandas()
    return train, test


if TRAIN_FILE is None:
    train_df, test_df = load_demo_data()
else:
    train_df = pd.read_csv(TRAIN_FILE).rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
    test_df = pd.read_csv(TEST_FILE).rename(columns={TEXT_COL: "text", LABEL_COL: "label"})

train_df = train_df.dropna(subset=["text", "label"])
train_df.head()

Xem phân phối nhãn (có bị lệch không?) và độ dài câu (để chọn `MAX_LEN`).

In [ ]:
print(train_df["label"].value_counts())
print()
print(train_df["text"].str.split().str.len().describe())

Đổi nhãn sang số 0, 1, 2, ...

In [ ]:
labels = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)
num_labels = len(labels)
print(label2id)

## 3. Làm sạch văn bản

In [ ]:
def normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"http\S+", " ", text)          # bỏ link
    text = re.sub(r"\s+", " ", text)
    return text.strip()


train_df["text"] = train_df["text"].apply(normalize)
test_df["text"] = test_df["text"].apply(normalize)

In [ ]:
if USE_WORD_SEG:
    from pyvi import ViTokenizer
    train_df["text"] = train_df["text"].apply(ViTokenizer.tokenize)
    test_df["text"] = test_df["text"].apply(ViTokenizer.tokenize)

Chia 90% train, 10% valid (giữ tỉ lệ nhãn bằng `stratify`).

In [ ]:
train_part, valid_part = train_test_split(
    train_df, test_size=0.1, stratify=train_df["label_id"], random_state=42
)
print(len(train_part), len(valid_part))

## 4. Baseline: TF-IDF + Logistic Regression
Chạy trong vài giây, không cần GPU. **Luôn làm bước này trước.**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = vectorizer.fit_transform(train_part["text"])
X_valid = vectorizer.transform(valid_part["text"])

logreg = LogisticRegression(max_iter=2000, class_weight="balanced")
logreg.fit(X_train, train_part["label_id"])

valid_pred = logreg.predict(X_valid)
print("TF-IDF macro F1:", f1_score(valid_part["label_id"], valid_pred, average="macro"))

## 5. Fine-tune Transformer

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)


train_ds = Dataset.from_pandas(train_part[["text", "label_id"]].rename(columns={"label_id": "labels"}), preserve_index=False)
valid_ds = Dataset.from_pandas(valid_part[["text", "label_id"]].rename(columns={"label_id": "labels"}), preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["text"]], preserve_index=False)

train_ds = train_ds.map(tokenize, batched=True)
valid_ds = valid_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return {
        "f1": f1_score(labels, predictions, average="macro"),
        "accuracy": accuracy_score(labels, predictions),
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
)

args = TrainingArguments(
    output_dir="cls_output",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 6. Đánh giá trên valid

In [ ]:
valid_logits = trainer.predict(valid_ds).predictions
valid_pred = valid_logits.argmax(axis=-1)

print(classification_report(valid_part["label_id"], valid_pred, target_names=[str(l) for l in labels]))

## 7. Kết hợp TF-IDF + Transformer
Cộng xác suất của 2 model, thử vài trọng số trên valid.

In [ ]:
def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


prob_transformer = softmax(valid_logits)
prob_tfidf = logreg.predict_proba(X_valid)

for w in [0.0, 0.2, 0.3, 0.5]:
    prob = w * prob_tfidf + (1 - w) * prob_transformer
    f1 = f1_score(valid_part["label_id"], prob.argmax(axis=1), average="macro")
    print(f"trọng số TF-IDF = {w}: F1 = {f1:.4f}")

In [ ]:
W_TFIDF = 0.2   # sửa theo kết quả ô trên

## 8. Dự đoán test và lưu bài nộp

In [ ]:
test_prob = W_TFIDF * logreg.predict_proba(vectorizer.transform(test_df["text"]))
test_prob += (1 - W_TFIDF) * softmax(trainer.predict(test_ds).predictions)

test_df["prediction"] = [id2label[i] for i in test_prob.argmax(axis=1)]
test_df.to_csv("submission.csv", index=False)
test_df.head()

---
## 9. Nâng cao (nếu còn thời gian)
- **Model to hơn:** `xlm-roberta-large`, lr = 1e-5.
- **K-Fold:** train 5 model trên 5 cách chia, lấy trung bình xác suất → ổn định hơn.
- **Dữ liệu lệch nhãn:** dùng class weights (xem `knowledge/01`).
- **Multi-label:** thêm `problem_type="multi_label_classification"` khi load model, nhãn là vector 0/1.
- **Không có dữ liệu train:** dùng zero-shot (ô dưới).

In [ ]:
from transformers import pipeline

zero_shot = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")

zero_shot(
    "Giảng viên dạy rất nhiệt tình",
    candidate_labels=["tích cực", "tiêu cực", "trung tính"],
    hypothesis_template="Câu này mang ý nghĩa {}.",
)